# Seed-Robustness Check: SAC-Discrete and PEARL-style vs Belief+periodicity

`paper2_sac_discrete.ipynb` and `paper3_pearl_meta_context.ipynb` each report a single-seed,
25-scenario result:

- SAC-Discrete (residual, warm-started): **0.2293** hit rate, beating plain Belief-index (0.2133)
- PEARL-style (context-conditioned): **0.1885**, below both

Neither number has the seed-variance and significance treatment `rigorous_scheduler_comparison.ipynb`
already gave the bandit/Whittle family, and neither was compared against **Belief+periodicity**
(0.1887 in that notebook's 8-seed sweep) -- only plain Belief-index. This notebook closes both
gaps: **retrain SAC-Discrete and PEARL-style from scratch across 8 seeds** (fresh network
initialization and fresh domain-randomized training data each time, using the exact stabilized
recipe from Papers 2/3 -- Q-warmup, frozen index-weighted warm-start, bounded correction), and
run paired bootstrap significance against Belief+periodicity specifically.

**Scope note:** this reuses the Papers 2/3 training recipe verbatim (same hyperparameters,
same warm-start/Q-warmup fix) -- the question here is only "does the single-seed ranking
survive seed variance", not "can the recipe be improved further".

## 1. Environment, emitters, belief schedulers (reused unchanged)

In [ ]:
import math
import time
from collections import deque
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

N_BANDS = 12
BAND_WIDTH_MHZ = 200.0
BAND_EDGES_MHZ = np.arange(N_BANDS + 1) * BAND_WIDTH_MHZ + 2000.0
EPISODE_SLOTS = 150
SLOT_DURATION_US = 2000.0
PD_SENSOR = 0.9
PFA_SENSOR = 0.05
SENSITIVITY_DB = 8.0
SLOPE_DB = 3.0
FEATS_PER_BAND = 4
FEATS_DIM = N_BANDS * FEATS_PER_BAND
Z_DIM = 8
CONTEXT_TRANSITION_DIM = FEATS_DIM + N_BANDS + 1

print(f"{N_BANDS} bands x {EPISODE_SLOTS} slots, z_dim={Z_DIM}")

In [ ]:
@dataclass
class PDW:
    toa_us: float
    freq_mhz: float
    pw_us: float
    aoa_deg: float
    amp_db: float
    emitter_id: int


class Emitter:
    kind = "base"
    def __init__(self, emitter_id, band, pw_us=1.0, amp_db=8.0):
        self.emitter_id = emitter_id; self.band = band; self.pw_us = pw_us; self.amp_db = amp_db
    def simulate(self, n_slots, seed):
        raise NotImplementedError


class BurstyEmitter(Emitter):
    kind = "bursty"
    def __init__(self, *args, p_off_to_on=0.08, p_on_to_on=0.85, **kwargs):
        super().__init__(*args, **kwargs)
        self.p_off_to_on = p_off_to_on; self.p_on_to_on = p_on_to_on
    def simulate(self, n_slots, seed):
        r = np.random.default_rng(seed)
        on = np.zeros(n_slots, dtype=bool)
        state = r.random() < 0.3
        for t in range(n_slots):
            on[t] = state
            p = self.p_on_to_on if state else self.p_off_to_on
            state = r.random() < p
        return on, np.full(n_slots, self.band, dtype=int)


class PeriodicScanEmitter(Emitter):
    kind = "periodic"
    def __init__(self, *args, scan_period_slots=20, dwell_slots=3, phase=0, **kwargs):
        super().__init__(*args, **kwargs)
        self.scan_period_slots = scan_period_slots; self.dwell_slots = dwell_slots; self.phase = phase
    def simulate(self, n_slots, seed):
        t = np.arange(n_slots)
        on = ((t - self.phase) % self.scan_period_slots) < self.dwell_slots
        return on, np.full(n_slots, self.band, dtype=int)


class FrequencyHopEmitter(Emitter):
    kind = "hopper"
    def __init__(self, emitter_id, bands, hop_slots=4, on_prob=0.9, **kwargs):
        super().__init__(emitter_id, band=bands[0], **kwargs)
        self.bands = list(bands); self.hop_slots = hop_slots; self.on_prob = on_prob
    def simulate(self, n_slots, seed):
        r = np.random.default_rng(seed)
        on = r.random(n_slots) < self.on_prob
        n_hops = n_slots // self.hop_slots + 1
        hop_choices = r.integers(0, len(self.bands), size=n_hops)
        band_seq = np.array([self.bands[hop_choices[t // self.hop_slots]] for t in range(n_slots)])
        return on, band_seq


def make_random_scenario(n_bands=N_BANDS, n_slots=EPISODE_SLOTS, seed=0):
    r = np.random.default_rng(seed)
    emitters, eid = [], 0
    for _ in range(r.integers(1, 3)):
        band = int(r.integers(0, n_bands))
        emitters.append(BurstyEmitter(eid, band, amp_db=float(r.uniform(2, 15)),
                                       p_off_to_on=float(r.uniform(0.04, 0.15)),
                                       p_on_to_on=float(r.uniform(0.7, 0.95))))
        eid += 1
    for _ in range(r.integers(1, 3)):
        band = int(r.integers(0, n_bands))
        emitters.append(PeriodicScanEmitter(eid, band, amp_db=float(r.uniform(2, 15)),
                                             scan_period_slots=int(r.integers(12, 30)),
                                             dwell_slots=int(r.integers(2, 4)),
                                             phase=int(r.integers(0, n_slots))))
        eid += 1
    if r.random() < 0.8:
        n_hop_bands = int(r.integers(3, 6))
        bands = list(r.choice(n_bands, size=n_hop_bands, replace=False))
        emitters.append(FrequencyHopEmitter(eid, bands, amp_db=float(r.uniform(2, 15)),
                                             hop_slots=int(r.integers(2, 6)),
                                             on_prob=float(r.uniform(0.6, 0.95))))
        eid += 1
    return emitters


def build_truth_and_pdws(emitters, n_bands=N_BANDS, n_slots=EPISODE_SLOTS,
                          slot_us=SLOT_DURATION_US, band_edges=BAND_EDGES_MHZ, seed=0):
    truth = np.zeros((n_bands, n_slots), dtype=bool)
    owner = np.full((n_bands, n_slots), -1, dtype=int)
    amp_grid = np.full((n_bands, n_slots), -np.inf)
    r = np.random.default_rng(seed + 10_000)
    for e in emitters:
        on, band_seq = e.simulate(n_slots, seed=seed * 1000 + e.emitter_id)
        for t in range(n_slots):
            if not on[t]:
                continue
            b = int(band_seq[t])
            truth[b, t] = True
            amp_grid[b, t] = max(amp_grid[b, t], e.amp_db)
            if owner[b, t] == -1:
                owner[b, t] = e.emitter_id
    return truth, owner, amp_grid


class ScanEnv:
    def __init__(self, truth, amp_grid=None, sensitivity_db=SENSITIVITY_DB, slope_db=SLOPE_DB,
                 pd_fallback=PD_SENSOR, pfa=PFA_SENSOR, seed=0):
        self.truth = truth; self.amp_grid = amp_grid
        self.sensitivity_db = sensitivity_db; self.slope_db = slope_db
        self.pd_fallback = pd_fallback; self.pfa = pfa
        self.rng = np.random.default_rng(seed)
        self.n_bands, self.n_slots = truth.shape
        self.reset()

    def reset(self):
        self.t = 0
        self.belief = np.full(self.n_bands, 0.5)
        self.last_visit = np.full(self.n_bands, -1)
        self.hit_count = np.zeros(self.n_bands, dtype=int)
        self.visit_count = np.zeros(self.n_bands, dtype=int)
        self.trans_counts = np.ones((self.n_bands, 2, 2))
        return self._obs()

    def _obs(self):
        staleness = np.clip(self.t - self.last_visit, 0, None) / max(self.n_slots, 1)
        return {"belief": self.belief.copy(), "staleness": staleness, "t": self.t}

    def transition_probs(self, band):
        c = self.trans_counts[band]
        return c[0, 1] / c[0].sum(), c[1, 1] / c[1].sum()

    def _pd_for(self, action):
        if self.amp_grid is None:
            return self.pd_fallback
        amp = self.amp_grid[action, self.t]
        return float(1.0 / (1.0 + math.exp(-(amp - self.sensitivity_db) / self.slope_db)))

    def step(self, action: int):
        true_state = bool(self.truth[action, self.t])
        amp_db = None
        if true_state:
            pd_eff = self._pd_for(action)
            detect = self.rng.random() < pd_eff
            if self.amp_grid is not None:
                amp_db = float(self.amp_grid[action, self.t])
        else:
            detect = self.rng.random() < self.pfa
        reward = 1.0 if detect else 0.0
        self.visit_count[action] += 1
        self.hit_count[action] += int(detect)
        prior = self.belief[action]
        like_on = self.pd_fallback if detect else (1 - self.pd_fallback)
        like_off = self.pfa if detect else (1 - self.pfa)
        post = (like_on * prior) / (like_on * prior + like_off * (1 - prior) + 1e-9)
        self.belief[action] = post
        self.trans_counts[action, int(prior > 0.5), int(post > 0.5)] += 1
        self.last_visit[action] = self.t
        for b in range(self.n_bands):
            p01, p11 = self.transition_probs(b)
            bel = self.belief[b]
            self.belief[b] = bel * p11 + (1 - bel) * p01
        self.t += 1
        done = self.t >= self.n_slots
        info = {"true_state": true_state, "detect": detect, "amp_db": amp_db}
        return self._obs(), reward, done, info


class PeriodicityTracker:
    def __init__(self, n_bands, min_hits=3):
        self.n_bands = n_bands; self.min_hits = min_hits
        self.hit_times = [[] for _ in range(n_bands)]
    def record_hit(self, band, t):
        self.hit_times[band].append(t)
    def estimate_period(self, band):
        hits = self.hit_times[band]
        if len(hits) < self.min_hits:
            return None
        diffs = np.diff(hits[-6:])
        return float(np.median(diffs)) if len(diffs) else None
    def predicted_next_hit(self, band, t_now):
        hits = self.hit_times[band]
        period = self.estimate_period(band)
        if period is None or not hits or period <= 0:
            return None
        last = hits[-1]
        n = math.ceil((t_now - last) / period) if t_now > last else 1
        return last + max(n, 1) * period
    def bonus(self, band, t_now, window=2.0):
        pred = self.predicted_next_hit(band, t_now)
        if pred is None:
            return 0.0
        return math.exp(-abs(pred - t_now) / window)


def featurize(env, tracker):
    belief = env.belief
    staleness = np.clip(env.t - env.last_visit, 0, None) / max(env.n_slots, 1)
    hit_rate = env.hit_count / np.maximum(env.visit_count, 1)
    bonus = np.array([tracker.bonus(b, env.t) for b in range(env.n_bands)])
    return np.stack([belief, staleness, hit_rate, bonus], axis=1).astype(np.float32)


def staleness_bonus(env):
    return np.clip(env.t - env.last_visit, 0, None) / max(env.n_slots, 1)


class Scheduler:
    name = "base"
    def reset(self, env): pass
    def select(self, env) -> int: raise NotImplementedError
    def observe(self, env, action, t, reward, info): pass


class RoundRobinScheduler(Scheduler):
    name = "Round-robin"
    def reset(self, env): self._next = 0
    def select(self, env):
        a = self._next % env.n_bands
        self._next += 1
        return a


class GreedyBeliefScheduler(Scheduler):
    name = "Belief-index"
    def __init__(self, explore_weight=0.15):
        self.explore_weight = explore_weight
    def select(self, env):
        idx = env.belief + self.explore_weight * staleness_bonus(env)
        return int(np.argmax(idx))


class BeliefPeriodicityScheduler(Scheduler):
    name = "Belief+periodicity"
    def __init__(self, periodicity_weight=0.6, explore_weight=0.15):
        self.periodicity_weight = periodicity_weight
        self.explore_weight = explore_weight
    def reset(self, env):
        self.tracker = PeriodicityTracker(env.n_bands)
    def select(self, env):
        periodicity = np.array([self.tracker.bonus(b, env.t) for b in range(env.n_bands)])
        idx = env.belief + self.explore_weight * staleness_bonus(env) + self.periodicity_weight * periodicity
        return int(np.argmax(idx))
    def observe(self, env, action, t, reward, info):
        if info["detect"]:
            self.tracker.record_hit(action, t)


def make_scenarios(n, base_seed):
    scenarios = []
    for i in range(n):
        seed = base_seed + i
        emitters = make_random_scenario(seed=seed)
        truth, owner, amp_grid = build_truth_and_pdws(emitters, seed=seed)
        scenarios.append((truth, owner, amp_grid))
    return scenarios


def evaluate_scheduler(scheduler, scenarios, eval_seed_base=9000):
    total_scans = total_hits = 0
    true_on_scans = hits_when_on = true_off_scans = false_alarms = 0
    per_scenario_hit_rate = []
    for si, (truth, owner, amp_grid) in enumerate(scenarios):
        env = ScanEnv(truth, amp_grid=amp_grid, seed=eval_seed_base + si)
        scheduler.reset(env)
        scen_scans = scen_hits = 0
        for _ in range(env.n_slots):
            t = env.t
            action = scheduler.select(env)
            obs, reward, done, info = env.step(action)
            scheduler.observe(env, action, t, reward, info)
            total_scans += 1; scen_scans += 1
            if info["detect"]:
                total_hits += 1; scen_hits += 1
            if info["true_state"]:
                true_on_scans += 1; hits_when_on += int(info["detect"])
            else:
                true_off_scans += 1; false_alarms += int(info["detect"])
            if done:
                break
        per_scenario_hit_rate.append(scen_hits / max(scen_scans, 1))
    return {
        "scheduler": scheduler.name,
        "hit_rate": total_hits / max(total_scans, 1),
        "Pd": hits_when_on / max(true_on_scans, 1),
        "Pfa": false_alarms / max(true_off_scans, 1),
        "per_scenario_hit_rate": per_scenario_hit_rate,
    }


def paired_bootstrap_ci(diffs, n_boot=10000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    diffs = np.asarray(diffs)
    n = len(diffs)
    boot_means = np.array([diffs[rng.integers(0, n, n)].mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot_means, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return float(diffs.mean()), float(lo), float(hi)

## 2. SAC-Discrete and PEARL-style, exact Papers 2/3 recipe

Unchanged from the source notebooks: residual/warm-started policy (frozen belief-index
weights + bounded correction), Q-network warm-up before the policy is allowed to move,
`target_entropy = 0.2 * log(|A|)` (the fix for the diverging-temperature failure Paper 2
documents). PEARL adds the permutation-invariant Gaussian-product context encoder, sampled
via reparameterization, context batches drawn from a separate recent-data pool (PEARL's own
ablation: sampling context from the whole buffer hurts).

In [ ]:
class QNetSAC(nn.Module):
    def __init__(self, n_bands, feats_per_band=FEATS_PER_BAND, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_bands * feats_per_band, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_bands),
        )
    def forward(self, x):
        return self.net(x)


class ResidualPolicyNetSAC(nn.Module):
    def __init__(self, n_bands, feats_per_band=FEATS_PER_BAND, hidden=128):
        super().__init__()
        self.n_bands = n_bands
        self.feats_per_band = feats_per_band
        self.trunk = nn.Sequential(
            nn.Linear(n_bands * feats_per_band, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.correction_head = nn.Linear(hidden, n_bands)
        nn.init.zeros_(self.correction_head.weight)
        nn.init.zeros_(self.correction_head.bias)
        self.index_weights = nn.Parameter(torch.tensor([1.0, 0.15, 0.0, 0.6]))
        self.index_weights.requires_grad_(False)

    def forward(self, x, max_correction=0.5):
        batch = x.shape[0]
        x_bands = x.view(batch, self.n_bands, self.feats_per_band)
        index_score = (x_bands * self.index_weights).sum(dim=-1)
        h = self.trunk(x)
        correction = max_correction * torch.tanh(self.correction_head(h))
        logits = index_score + correction
        probs = F.softmax(logits, dim=-1)
        log_probs = F.log_softmax(logits, dim=-1)
        return probs, log_probs


class SACScheduler(Scheduler):
    name = "SAC-Discrete (residual, warm-started)"
    def __init__(self, policy, n_bands, deterministic=True):
        self.policy = policy
        self.n_bands = n_bands
        self.deterministic = deterministic
    def reset(self, env):
        self.tracker = PeriodicityTracker(env.n_bands)
    def select(self, env):
        feats = featurize(env, self.tracker)
        x = torch.from_numpy(feats.reshape(1, -1))
        with torch.no_grad():
            probs, _ = self.policy(x)
            action = torch.argmax(probs, dim=1) if self.deterministic else torch.distributions.Categorical(probs=probs).sample()
        return int(action.item())
    def observe(self, env, action, t, reward, info):
        if info["detect"]:
            self.tracker.record_hit(action, t)


def soft_update(target, source, tau):
    with torch.no_grad():
        for tp, sp in zip(target.parameters(), source.parameters()):
            tp.data.mul_(1 - tau).add_(tau * sp.data)


def train_sac_discrete(n_bands=N_BANDS, n_slots=EPISODE_SLOTS, total_steps=20000,
                        replay_size=20000, batch_size=128, gamma=0.99, lr=3e-4, tau=0.005,
                        start_steps=1000, seed=0, target_entropy_frac=0.2, q_warmup_steps=3000):
    torch.manual_seed(seed)
    q1, q2 = QNetSAC(n_bands), QNetSAC(n_bands)
    q1_targ, q2_targ = QNetSAC(n_bands), QNetSAC(n_bands)
    q1_targ.load_state_dict(q1.state_dict()); q2_targ.load_state_dict(q2.state_dict())
    for p in q1_targ.parameters(): p.requires_grad_(False)
    for p in q2_targ.parameters(): p.requires_grad_(False)
    policy = ResidualPolicyNetSAC(n_bands)

    target_entropy = target_entropy_frac * math.log(n_bands)
    log_alpha = torch.zeros(1, requires_grad=True)
    q_opt = torch.optim.Adam(list(q1.parameters()) + list(q2.parameters()), lr=lr)
    pi_opt = torch.optim.Adam(list(policy.trunk.parameters()) + list(policy.correction_head.parameters()), lr=lr)
    alpha_opt = torch.optim.Adam([log_alpha], lr=lr)

    buffer = deque(maxlen=replay_size)
    master_rng = np.random.default_rng(seed)
    env = tracker = None
    t_in_ep = 0
    step = 0

    while step < total_steps:
        if env is None or t_in_ep >= n_slots:
            scenario_seed = int(master_rng.integers(0, 1_000_000))
            emitters = make_random_scenario(n_bands=n_bands, n_slots=n_slots, seed=scenario_seed)
            truth, owner, amp_grid = build_truth_and_pdws(emitters, n_bands=n_bands, n_slots=n_slots, seed=scenario_seed)
            env = ScanEnv(truth, amp_grid=amp_grid, seed=scenario_seed + 500_000)
            tracker = PeriodicityTracker(env.n_bands)
            env.reset()
            t_in_ep = 0

        feats = featurize(env, tracker)
        if step < start_steps:
            action = int(np.random.randint(n_bands))
        else:
            x = torch.from_numpy(feats.reshape(1, -1))
            with torch.no_grad():
                probs, _ = policy(x)
                action = int(torch.distributions.Categorical(probs=probs).sample().item())

        obs, reward, done, info = env.step(action)
        if info["detect"]:
            tracker.record_hit(action, env.t - 1)
        next_feats = featurize(env, tracker)
        buffer.append((feats, action, reward, next_feats, float(done)))
        t_in_ep += 1
        step += 1

        if step >= start_steps and len(buffer) >= batch_size:
            idx = np.random.randint(0, len(buffer), size=batch_size)
            batch = [buffer[i] for i in idx]
            s = torch.from_numpy(np.stack([b[0] for b in batch]).reshape(batch_size, -1)).float()
            a = torch.tensor([b[1] for b in batch], dtype=torch.long)
            r = torch.tensor([b[2] for b in batch], dtype=torch.float32)
            s2 = torch.from_numpy(np.stack([b[3] for b in batch]).reshape(batch_size, -1)).float()
            d = torch.tensor([b[4] for b in batch], dtype=torch.float32)
            alpha = log_alpha.exp()

            with torch.no_grad():
                next_probs, next_log_probs = policy(s2)
                min_q_t = torch.min(q1_targ(s2), q2_targ(s2))
                v_next = (next_probs * (min_q_t - alpha * next_log_probs)).sum(dim=1)
                target = r + gamma * (1 - d) * v_next

            q1_pred = q1(s).gather(1, a.unsqueeze(1)).squeeze(1)
            q2_pred = q2(s).gather(1, a.unsqueeze(1)).squeeze(1)
            q_loss = F.mse_loss(q1_pred, target) + F.mse_loss(q2_pred, target)
            q_opt.zero_grad(); q_loss.backward(); q_opt.step()
            soft_update(q1_targ, q1, tau); soft_update(q2_targ, q2, tau)

            if step >= q_warmup_steps:
                probs, log_probs = policy(s)
                with torch.no_grad():
                    min_q_s = torch.min(q1(s), q2(s))
                pi_loss = (probs * (alpha.detach() * log_probs - min_q_s)).sum(dim=1).mean()
                pi_opt.zero_grad(); pi_loss.backward(); pi_opt.step()
                alpha_loss = (probs.detach() * (-log_alpha.exp() * (log_probs.detach() + target_entropy))).sum(dim=1).mean()
                alpha_opt.zero_grad(); alpha_loss.backward(); alpha_opt.step()

        if done:
            env = None

    return SACScheduler(policy, n_bands, deterministic=True)

In [ ]:
def transition_to_context_row(feats_flat, action, reward, n_bands):
    action_onehot = np.zeros(n_bands, dtype=np.float32)
    action_onehot[action] = 1.0
    return np.concatenate([feats_flat, action_onehot, [reward]]).astype(np.float32)


class ContextEncoder(nn.Module):
    def __init__(self, transition_dim=CONTEXT_TRANSITION_DIM, z_dim=Z_DIM, hidden=64):
        super().__init__()
        self.z_dim = z_dim
        self.net = nn.Sequential(
            nn.Linear(transition_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.mu_head = nn.Linear(hidden, z_dim)
        self.logvar_head = nn.Linear(hidden, z_dim)

    def forward(self, context):
        if context.shape[0] == 0:
            return torch.zeros(self.z_dim), torch.ones(self.z_dim)
        h = self.net(context)
        mu_per = self.mu_head(h)
        var_per = torch.exp(self.logvar_head(h)).clamp(min=1e-4)
        precision_per = 1.0 / var_per
        precision_total = precision_per.sum(dim=0) + 1.0
        mu_total = (precision_per * mu_per).sum(dim=0) / precision_total
        var_total = 1.0 / precision_total
        return mu_total, var_total


def sample_z(mu, var):
    return mu + torch.randn_like(mu) * torch.sqrt(var + 1e-6)


def kl_to_prior(mu, var):
    return 0.5 * (var + mu.pow(2) - 1 - torch.log(var + 1e-6)).sum()


class QNetPEARL(nn.Module):
    def __init__(self, n_bands, feats_dim=FEATS_DIM, z_dim=Z_DIM, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feats_dim + z_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_bands),
        )
    def forward(self, x, z):
        return self.net(torch.cat([x, z], dim=-1))


class ContextResidualPolicyNet(nn.Module):
    def __init__(self, n_bands, feats_per_band=FEATS_PER_BAND, z_dim=Z_DIM, hidden=128):
        super().__init__()
        self.n_bands = n_bands
        self.feats_per_band = feats_per_band
        self.trunk = nn.Sequential(
            nn.Linear(n_bands * feats_per_band + z_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.correction_head = nn.Linear(hidden, n_bands)
        nn.init.zeros_(self.correction_head.weight)
        nn.init.zeros_(self.correction_head.bias)
        self.index_weights = nn.Parameter(torch.tensor([1.0, 0.15, 0.0, 0.6]))
        self.index_weights.requires_grad_(False)

    def forward(self, x, z, max_correction=0.5):
        batch = x.shape[0]
        x_bands = x.view(batch, self.n_bands, self.feats_per_band)
        index_score = (x_bands * self.index_weights).sum(dim=-1)
        h = self.trunk(torch.cat([x, z], dim=-1))
        correction = max_correction * torch.tanh(self.correction_head(h))
        logits = index_score + correction
        probs = F.softmax(logits, dim=-1)
        log_probs = F.log_softmax(logits, dim=-1)
        return probs, log_probs


class PEARLScheduler(Scheduler):
    name = "PEARL-style (context-conditioned)"
    def __init__(self, policy, encoder, n_bands, context_window=20, deterministic=True):
        self.policy = policy
        self.encoder = encoder
        self.n_bands = n_bands
        self.context_window = context_window
        self.deterministic = deterministic
    def reset(self, env):
        self.tracker = PeriodicityTracker(env.n_bands)
        self.context_buffer = deque(maxlen=self.context_window)
    def _current_z(self):
        if len(self.context_buffer) == 0:
            ctx = torch.zeros((0, CONTEXT_TRANSITION_DIM))
        else:
            ctx = torch.from_numpy(np.stack(list(self.context_buffer)))
        with torch.no_grad():
            mu, var = self.encoder(ctx)
        return mu.unsqueeze(0)
    def select(self, env):
        feats = featurize(env, self.tracker)
        feats_flat = feats.reshape(-1)
        x = torch.from_numpy(feats_flat.reshape(1, -1))
        z = self._current_z()
        with torch.no_grad():
            probs, _ = self.policy(x, z)
            action = torch.argmax(probs, dim=1) if self.deterministic else torch.distributions.Categorical(probs=probs).sample()
        action = int(action.item())
        self._last_feats_flat = feats_flat
        return action
    def observe(self, env, action, t, reward, info):
        if info["detect"]:
            self.tracker.record_hit(action, t)
        row = transition_to_context_row(self._last_feats_flat, action, reward, self.n_bands)
        self.context_buffer.append(row)


def train_pearl(n_bands=N_BANDS, n_slots=EPISODE_SLOTS, total_steps=25000,
                 replay_size=20000, context_pool_size=2000, batch_size=128, context_batch_size=32,
                 gamma=0.99, lr=3e-4, tau=0.005, start_steps=1000, seed=0,
                 target_entropy_frac=0.2, q_warmup_steps=3000, context_window=20, kl_weight=0.01):
    torch.manual_seed(seed)
    q1, q2 = QNetPEARL(n_bands), QNetPEARL(n_bands)
    q1_targ, q2_targ = QNetPEARL(n_bands), QNetPEARL(n_bands)
    q1_targ.load_state_dict(q1.state_dict()); q2_targ.load_state_dict(q2.state_dict())
    for p in q1_targ.parameters(): p.requires_grad_(False)
    for p in q2_targ.parameters(): p.requires_grad_(False)
    policy = ContextResidualPolicyNet(n_bands)
    encoder = ContextEncoder()

    target_entropy = target_entropy_frac * math.log(n_bands)
    log_alpha = torch.zeros(1, requires_grad=True)
    q_opt = torch.optim.Adam(list(q1.parameters()) + list(q2.parameters()) + list(encoder.parameters()), lr=lr)
    pi_opt = torch.optim.Adam(list(policy.trunk.parameters()) + list(policy.correction_head.parameters()), lr=lr)
    alpha_opt = torch.optim.Adam([log_alpha], lr=lr)

    buffer = deque(maxlen=replay_size)
    context_pool = deque(maxlen=context_pool_size)
    master_rng = np.random.default_rng(seed)
    env = tracker = ep_context = None
    t_in_ep = 0
    step = 0

    while step < total_steps:
        if env is None or t_in_ep >= n_slots:
            scenario_seed = int(master_rng.integers(0, 1_000_000))
            emitters = make_random_scenario(n_bands=n_bands, n_slots=n_slots, seed=scenario_seed)
            truth, owner, amp_grid = build_truth_and_pdws(emitters, n_bands=n_bands, n_slots=n_slots, seed=scenario_seed)
            env = ScanEnv(truth, amp_grid=amp_grid, seed=scenario_seed + 500_000)
            tracker = PeriodicityTracker(env.n_bands)
            env.reset()
            ep_context = deque(maxlen=context_window)
            t_in_ep = 0

        feats = featurize(env, tracker)
        feats_flat = feats.reshape(-1)
        if len(ep_context) == 0:
            z_np = np.zeros(Z_DIM, dtype=np.float32)
        else:
            ctx_t = torch.from_numpy(np.stack(list(ep_context)))
            with torch.no_grad():
                mu, var = encoder(ctx_t)
                z_np = sample_z(mu, var).numpy()

        if step < start_steps:
            action = int(np.random.randint(n_bands))
        else:
            x = torch.from_numpy(feats_flat.reshape(1, -1))
            z = torch.from_numpy(z_np.reshape(1, -1))
            with torch.no_grad():
                probs, _ = policy(x, z)
                action = int(torch.distributions.Categorical(probs=probs).sample().item())

        obs, reward, done, info = env.step(action)
        if info["detect"]:
            tracker.record_hit(action, env.t - 1)
        next_feats = featurize(env, tracker).reshape(-1)
        buffer.append((feats_flat, action, reward, next_feats, float(done)))
        ctx_row = transition_to_context_row(feats_flat, action, reward, n_bands)
        ep_context.append(ctx_row)
        context_pool.append(ctx_row)
        t_in_ep += 1
        step += 1

        if step >= start_steps and len(buffer) >= batch_size and len(context_pool) >= context_batch_size:
            idx = np.random.randint(0, len(buffer), size=batch_size)
            batch = [buffer[i] for i in idx]
            s = torch.from_numpy(np.stack([b[0] for b in batch])).float()
            a = torch.tensor([b[1] for b in batch], dtype=torch.long)
            r = torch.tensor([b[2] for b in batch], dtype=torch.float32)
            s2 = torch.from_numpy(np.stack([b[3] for b in batch])).float()
            d = torch.tensor([b[4] for b in batch], dtype=torch.float32)

            ctx_idx = np.random.choice(len(context_pool), size=context_batch_size, replace=False)
            ctx_batch = torch.from_numpy(np.stack([context_pool[i] for i in ctx_idx])).float()
            mu, var = encoder(ctx_batch)
            z = sample_z(mu, var)
            z_batch = z.unsqueeze(0).expand(batch_size, -1)
            kl = kl_to_prior(mu, var)

            alpha = log_alpha.exp()
            with torch.no_grad():
                next_probs, next_log_probs = policy(s2, z_batch)
                min_q_t = torch.min(q1_targ(s2, z_batch), q2_targ(s2, z_batch))
                v_next = (next_probs * (min_q_t - alpha * next_log_probs)).sum(dim=1)
                target = r + gamma * (1 - d) * v_next

            q1_pred = q1(s, z_batch).gather(1, a.unsqueeze(1)).squeeze(1)
            q2_pred = q2(s, z_batch).gather(1, a.unsqueeze(1)).squeeze(1)
            q_loss = F.mse_loss(q1_pred, target) + F.mse_loss(q2_pred, target) + kl_weight * kl
            q_opt.zero_grad(); q_loss.backward(); q_opt.step()
            soft_update(q1_targ, q1, tau); soft_update(q2_targ, q2, tau)

            if step >= q_warmup_steps:
                z_batch_pi = z_batch.detach()
                probs, log_probs = policy(s, z_batch_pi)
                with torch.no_grad():
                    min_q_s = torch.min(q1(s, z_batch_pi), q2(s, z_batch_pi))
                pi_loss = (probs * (alpha.detach() * log_probs - min_q_s)).sum(dim=1).mean()
                pi_opt.zero_grad(); pi_loss.backward(); pi_opt.step()
                alpha_loss = (probs.detach() * (-log_alpha.exp() * (log_probs.detach() + target_entropy))).sum(dim=1).mean()
                alpha_opt.zero_grad(); alpha_loss.backward(); alpha_opt.step()

        if done:
            env = None

    return PEARLScheduler(policy, encoder, n_bands, context_window=context_window, deterministic=True)

## 3. 8-seed sweep

Each seed: fresh network initialization, fresh 20k-step SAC-Discrete training run, fresh
25k-step PEARL-style training run, fresh 25-scenario held-out evaluation set (disjoint from
that seed's training data), all five schedulers evaluated on the same scenarios. This is the
long step -- SAC-Discrete is ~3ms/step (~60s/seed), PEARL is somewhat heavier with the context
encoder (~100-120s/seed); 8 seeds of both plus evaluation is the bulk of this notebook's
runtime.

In [ ]:
t0 = time.time()
SEEDS = list(range(1, 9))
rows = []
per_scenario_by_scheduler = {}

for sd in SEEDS:
    np.random.seed(sd)
    eval_scenarios = make_scenarios(25, base_seed=777_000 + sd * 1000)

    sac_sched = train_sac_discrete(total_steps=20000, seed=sd)
    pearl_sched = train_pearl(total_steps=25000, seed=sd)

    schedulers = [
        RoundRobinScheduler(),
        GreedyBeliefScheduler(),
        BeliefPeriodicityScheduler(),
        sac_sched,
        pearl_sched,
    ]
    for s in schedulers:
        r = evaluate_scheduler(s, eval_scenarios)
        r["seed"] = sd
        rows.append(r)
        per_scenario_by_scheduler.setdefault(r["scheduler"], []).extend(r["per_scenario_hit_rate"])

    print(f"seed {sd}/{len(SEEDS)} done, elapsed {time.time()-t0:.1f}s", flush=True)

df = pd.DataFrame(rows)
print(f"\ntotal sweep time: {time.time()-t0:.1f}s")

## 4. Results: mean +/- std across 8 seeds

In [ ]:
summary = df.groupby("scheduler")[["hit_rate", "Pd", "Pfa"]].agg(["mean", "std"])
rank_order = df.groupby("scheduler")["hit_rate"].mean().sort_values(ascending=False).index
summary = summary.loc[rank_order]
summary.round(4)

In [ ]:
means = df.groupby("scheduler")["hit_rate"].mean().loc[rank_order]
stds = df.groupby("scheduler")["hit_rate"].std().loc[rank_order]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(range(len(means)), means.values, yerr=stds.values, capsize=4,
       color=plt.cm.tab10.colors[:len(means)])
ax.set_xticks(range(len(means)))
ax.set_xticklabels(means.index, rotation=25, ha="right")
ax.set_ylabel("hit rate (mean +/- std, 8 seeds x 25 scenarios)")
ax.set_title("SAC-Discrete and PEARL-style vs belief family, with seed variance")
plt.tight_layout()
plt.show()

## 5. Paired significance vs Belief+periodicity

Same paired-bootstrap method as `rigorous_scheduler_comparison.ipynb`: every scheduler in a
given seed faces the *same* 25 scenarios, so the per-`(seed, scenario)` hit-rate difference is
a valid paired sample (200 pairs total across 8 seeds).

In [ ]:
a = "Belief+periodicity"
comparisons = [s for s in rank_order if s != a]

sig_rows = []
for b in comparisons:
    diffs = np.array(per_scenario_by_scheduler[a]) - np.array(per_scenario_by_scheduler[b])
    mean, lo, hi = paired_bootstrap_ci(diffs)
    sig_rows.append({"comparison": f"{a}  -  {b}", "mean_diff": mean, "ci_lo": lo, "ci_hi": hi,
                      "significant (95%)": bool(lo > 0 or hi < 0)})

pd.DataFrame(sig_rows).round(4)

## 6. Conclusion

Compare Section 4's ranking (with error bars) and Section 5's significance table against the
two single-seed claims this notebook set out to check:

- **"SAC-Discrete (0.2293) beats Belief-index (0.2133)"** -- check whether that gap survives
  8-seed variance, and where SAC-Discrete lands relative to **Belief+periodicity** specifically
  (0.1887 in the earlier 8-seed sweep), which the original single-seed comparison never tested.
- **"PEARL-style (0.1885) underperforms both"** -- check whether that ordering is stable or
  seed-dependent.

Read the significance table's `significant (95%)` column plainly: a `True` means the 95%
bootstrap CI on the paired hit-rate difference excludes zero across all 8 seeds -- a real,
seed-robust effect, not a single-draw artifact. A `False` means the single-seed papers'
apparent ranking should be treated as within noise until more evidence accumulates.